# Conditional Forecasting

**Reference:** Waggoner, D.F. & Zha, T. (1999). "Conditional Forecasts in Dynamic Multivariate Models."
*Review of Economics and Statistics*, 81(4), 639-651.

---

In macroeconomic forecasting, we often want to answer questions like:
- *"What happens to GDP growth **if** the Fed raises rates to 5%?"*
- *"What is the inflation outlook **given** an assumed oil price path?"*

Standard (unconditional) forecasts let the model decide the future of all variables.
**Conditional forecasts** fix the trajectory of one or more variables and compute
the optimal forecast for the remaining variables, subject to those constraints.

The Waggoner-Zha (1999) algorithm finds the minimum-variance conditional forecast:

$$\hat{y}_{cond} = \hat{y}_{unc} + \Sigma_f R' (R \Sigma_f R')^{-1} (r - R \hat{y}_{unc})$$

where $R$ selects the constrained elements and $r$ contains the imposed values.

**Topics covered:**
1. Unconditional vs Conditional Forecast
2. Conditioning on a single variable (Fed Funds path)
3. Conditioning on multiple variables simultaneously
4. Soft conditioning (interval restrictions)
5. Comparing hawkish vs dovish scenarios

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from forecastbox.scenarios import ConditionalForecast, SimpleVAR, ScenarioBuilder

import sys
sys.path.insert(0, "..")
from utils.helpers import load_us_macro_quarterly, load_macro_brazil

%matplotlib inline
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["figure.dpi"] = 100

## 1. Unconditional vs Conditional Forecast

An **unconditional forecast** lets the VAR model project all variables freely based on
estimated dynamics. A **conditional forecast** imposes constraints on some variables'
future paths and adjusts the remaining forecasts optimally.

Let's start by loading the US macro quarterly data, estimating a VAR(2) model, and
generating an unconditional forecast as our baseline.

In [ ]:
# Load US macro quarterly data
df = load_us_macro_quarterly()
print("Shape:", df.shape)
print("Columns:", list(df.columns))
print("Period:", df.index[0].strftime("%Y-Q%q"), "to", df.index[-1].strftime("%Y-Q%q"))
df.tail()

# Variables: gdp_growth, inflation, fed_funds, unemployment

In [ ]:
# Estimate a VAR(2) model using SimpleVAR
endog = df[["gdp_growth", "inflation", "fed_funds", "unemployment"]].values
var_names = ["gdp_growth", "inflation", "fed_funds", "unemployment"]

model = SimpleVAR(endog, p_order=2, var_names=var_names)
print(f"VAR({model.p_order}) estimated with {model.k_vars} variables")
print(f"Observations: {endog.shape[0]}, Effective sample: {model.residuals.shape[0]}")
print(f"Variables: {model.var_names}")

# Generate unconditional forecast (no conditions)
cf = ConditionalForecast(model, method="analytic")
steps = 8
unc_forecast = cf.forecast(steps=steps, conditions=None, n_draws=1000, seed=42)

# Display unconditional point forecasts
unc_df = pd.DataFrame(
    {name: unc_forecast[name].point for name in var_names},
    index=[f"Q+{h+1}" for h in range(steps)]
)
print("\nUnconditional Forecast:")
unc_df.round(3)

## 2. Conditioning on Fed Funds Path

Now let's fix the trajectory of the federal funds rate and see how other variables
(GDP growth, inflation, unemployment) adjust. This is a **hard conditioning** constraint:
the model is forced to produce forecasts consistent with the imposed path.

We'll impose a gradual tightening path: the Fed raises rates by 25bp per quarter.

In [ ]:
# Conditioning on a rising Fed Funds path (tightening)
last_ff = df["fed_funds"].iloc[-1]
print(f"Last observed fed_funds: {last_ff:.2f}%")

# Impose: rate rises by 0.25pp each quarter for 8 quarters
ff_path = [last_ff + 0.25 * (h + 1) for h in range(steps)]
print(f"Imposed fed_funds path: {[round(x, 2) for x in ff_path]}")

# Conditional forecast with fed_funds fixed
cond_forecast = cf.forecast(
    steps=steps,
    conditions={"fed_funds": ff_path},
    n_draws=1000,
    seed=42,
)

# Compare unconditional vs conditional
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
horizons = np.arange(1, steps + 1)

for idx, var in enumerate(var_names):
    ax = axes[idx // 2, idx % 2]
    ax.plot(horizons, unc_forecast[var].point, "b-o", label="Unconditional", markersize=4)
    ax.plot(horizons, cond_forecast[var].point, "r-s", label="Conditional", markersize=4)

    # Show 80% CI for conditional
    if cond_forecast[var].lower_80 is not None:
        ax.fill_between(horizons, cond_forecast[var].lower_80, cond_forecast[var].upper_80,
                        alpha=0.15, color="red")

    ax.set_title(var, fontsize=12)
    ax.set_xlabel("Horizon (quarters)")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

fig.suptitle("Unconditional vs Conditional Forecast (Fed Funds Tightening)", fontsize=14)
fig.tight_layout()
plt.show()

## 3. Conditioning on Multiple Variables

We can impose conditions on **multiple variables simultaneously**. For example, we might
want to see the GDP and unemployment outlook given both a rising interest rate *and*
a specific inflation target path.

This is useful for policy analysis: "What if the Fed tightens rates while inflation
remains anchored at 2%?"

In [ ]:
# Condition on both fed_funds (tightening) and inflation (anchored at 2%)
inflation_target = [2.0] * steps  # inflation anchored at 2% for all 8 quarters

cond_multi = cf.forecast(
    steps=steps,
    conditions={
        "fed_funds": ff_path,
        "inflation": inflation_target,
    },
    n_draws=1000,
    seed=42,
)

# Compare: unconditional vs single-condition vs multi-condition
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for idx, var in enumerate(["gdp_growth", "unemployment"]):
    ax = axes[idx]
    ax.plot(horizons, unc_forecast[var].point, "b-o", label="Unconditional", markersize=4)
    ax.plot(horizons, cond_forecast[var].point, "r-s", label="Cond: fed_funds only", markersize=4)
    ax.plot(horizons, cond_multi[var].point, "g-^", label="Cond: fed_funds + inflation", markersize=4)

    ax.set_title(var, fontsize=12)
    ax.set_xlabel("Horizon (quarters)")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

fig.suptitle("Multi-Variable Conditioning: Fed Funds + Inflation Target", fontsize=14)
fig.tight_layout()
plt.show()

# Table: conditioned variables match their imposed paths exactly
print("Verification - Conditioned variables match imposed paths:")
print(f"  fed_funds imposed:  {[round(x, 2) for x in ff_path]}")
print(f"  fed_funds forecast: {[round(x, 2) for x in cond_multi['fed_funds'].point]}")
print(f"  inflation imposed:  {inflation_target}")
print(f"  inflation forecast: {[round(x, 2) for x in cond_multi['inflation'].point]}")

## 4. Soft Conditioning

**Hard conditioning** forces variables to match exact values. In practice, we may only
have a range or interval in mind (e.g., "the Fed will keep rates between 2% and 3%").

**Soft conditioning** can be approximated by running multiple scenarios within the
desired range and averaging, or by using the Gibbs sampler method which naturally
handles uncertainty around the imposed paths.

Below we demonstrate soft conditioning by:
1. Generating a grid of paths within the interval [2.0, 3.0] for fed_funds
2. Running conditional forecasts for each
3. Computing the envelope (mean and range) of resulting forecasts

In [ ]:
# Soft conditioning: fed_funds constrained to [2.0, 3.0] interval
# We sample N paths within the interval and average the conditional forecasts
rng = np.random.default_rng(42)
n_soft = 50  # number of sampled paths
soft_low, soft_high = 2.0, 3.0

# Store forecasts for each variable across all sampled paths
soft_results = {var: [] for var in var_names}

for i in range(n_soft):
    # Random path within the interval
    soft_path = rng.uniform(soft_low, soft_high, size=steps).tolist()
    result = cf.forecast(steps=steps, conditions={"fed_funds": soft_path}, n_draws=100, seed=i)
    for var in var_names:
        soft_results[var].append(result[var].point)

# Compute mean and range of soft-conditioned forecasts
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

for idx, var in enumerate(var_names):
    ax = axes[idx // 2, idx % 2]
    paths_array = np.array(soft_results[var])  # (n_soft, steps)

    mean_path = paths_array.mean(axis=0)
    low_path = paths_array.min(axis=0)
    high_path = paths_array.max(axis=0)
    q10 = np.quantile(paths_array, 0.10, axis=0)
    q90 = np.quantile(paths_array, 0.90, axis=0)

    ax.fill_between(horizons, low_path, high_path, alpha=0.1, color="green", label="Full range")
    ax.fill_between(horizons, q10, q90, alpha=0.25, color="green", label="80% envelope")
    ax.plot(horizons, mean_path, "g-o", label="Soft cond. mean", linewidth=2, markersize=4)
    ax.plot(horizons, unc_forecast[var].point, "b--", label="Unconditional", alpha=0.7)

    ax.set_title(var, fontsize=12)
    ax.set_xlabel("Horizon (quarters)")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

fig.suptitle(f"Soft Conditioning: Fed Funds in [{soft_low}, {soft_high}]", fontsize=14)
fig.tight_layout()
plt.show()

## 5. Comparing Conditional Scenarios

Let's build and compare three policy scenarios using the `ScenarioBuilder`:

- **Baseline**: No conditions (unconditional forecast)
- **Hawkish**: Fed aggressively raises rates (+50bp/quarter for 8 quarters)
- **Dovish**: Fed cuts rates (-25bp/quarter for 8 quarters, floored at 0)

In [ ]:
# Build scenarios using ScenarioBuilder
builder = ScenarioBuilder(model)

# Baseline: no conditions (empty dict triggers unconditional forecast)
# We use mild conditions close to unconditional to get a "baseline" through the builder
baseline_ff = unc_forecast["fed_funds"].point.tolist()
builder.add_scenario("baseline", {"fed_funds": baseline_ff}, description="Unconditional baseline path")

# Hawkish: aggressive tightening (+50bp/quarter)
hawkish_ff = [max(last_ff + 0.50 * (h + 1), 0.0) for h in range(steps)]
builder.add_scenario("hawkish", {"fed_funds": hawkish_ff}, description="Aggressive rate hikes (+50bp/quarter)")

# Dovish: rate cuts (-25bp/quarter, floor at 0)
dovish_ff = [max(last_ff - 0.25 * (h + 1), 0.0) for h in range(steps)]
builder.add_scenario("dovish", {"fed_funds": dovish_ff}, description="Gradual rate cuts (-25bp/quarter)")

print("Registered scenarios:", builder.list_scenarios())
print(f"\nHawkish fed_funds path: {[round(x, 2) for x in hawkish_ff]}")
print(f"Dovish fed_funds path:  {[round(x, 2) for x in dovish_ff]}")

# Run all scenarios
results = builder.run(steps=steps, n_draws=1000, seed=42)

# Plot comparison for each variable
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
colors = {"baseline": "blue", "hawkish": "red", "dovish": "green"}

for idx, var in enumerate(var_names):
    ax = axes[idx // 2, idx % 2]
    for scenario_name in ["baseline", "hawkish", "dovish"]:
        fc = results.get(scenario_name, var)
        ax.plot(horizons, fc.point, "-o", color=colors[scenario_name],
                label=scenario_name.capitalize(), linewidth=2, markersize=4)
        if fc.lower_80 is not None:
            ax.fill_between(horizons, fc.lower_80, fc.upper_80,
                            alpha=0.1, color=colors[scenario_name])

    ax.set_title(var, fontsize=12)
    ax.set_xlabel("Horizon (quarters)")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

fig.suptitle("Scenario Comparison: Baseline vs Hawkish vs Dovish", fontsize=14)
fig.tight_layout()
plt.show()

# Summary table
print("\n" + results.summary())

## Exercise 1: Condition on unemployment path (recession scenario)

Create a recession scenario by imposing a rising unemployment path (e.g., unemployment
increases by 0.5pp per quarter for 8 quarters). Examine how GDP growth, inflation,
and fed_funds respond to this labor market deterioration.

In [ ]:
# Exercise 1 - SOLUTION: Recession scenario with rising unemployment

# Define unemployment rising path: current level + 0.375pp per quarter
# This gives +3pp over 8 quarters
last_unemp = df["unemployment"].iloc[-1]
print(f"Last observed unemployment: {last_unemp:.2f}%")

recession_path = [last_unemp + (3.0 / steps) * (h + 1) for h in range(steps)]
print(f"Recession unemployment path: {[round(x, 2) for x in recession_path]}")
print(f"Total increase: {recession_path[-1] - last_unemp:.2f} pp over {steps} quarters")

# Conditional forecast with unemployment fixed to recession path
cond_recession = cf.forecast(
    steps=steps,
    conditions={"unemployment": recession_path},
    n_draws=1000,
    seed=42,
)

# Compare baseline (unconditional) vs recession scenario
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

for idx, var in enumerate(var_names):
    ax = axes[idx // 2, idx % 2]
    ax.plot(horizons, unc_forecast[var].point, "b-o", label="Baseline", markersize=4, linewidth=2)
    ax.plot(horizons, cond_recession[var].point, "r-s", label="Recession", markersize=4, linewidth=2)

    # Show 80% CI for recession
    if cond_recession[var].lower_80 is not None:
        ax.fill_between(horizons, cond_recession[var].lower_80, cond_recession[var].upper_80,
                        alpha=0.15, color="red")

    ax.set_title(var, fontsize=12)
    ax.set_xlabel("Horizon (quarters)")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

fig.suptitle("Recession Scenario: Unemployment Rising +3pp over 8 Quarters", fontsize=14)
fig.tight_layout()
plt.show()

# Summary table
print("\nRecession Scenario - Point Forecasts:")
recession_df = pd.DataFrame(
    {name: cond_recession[name].point for name in var_names},
    index=[f"Q+{h+1}" for h in range(steps)]
)
print(recession_df.round(3).to_string())

# Impact: difference from baseline
print("\nImpact (Recession - Baseline):")
impact_df = recession_df - unc_df
print(impact_df.round(3).to_string())

# Reference values
print("\n--- Reference Values ---")
print(f"Baseline GDP Q+1: {unc_forecast['gdp_growth'].point[0]:.3f}%")
print(f"Recession GDP Q+1: {cond_recession['gdp_growth'].point[0]:.3f}%")
print(f"Baseline GDP Q+8: {unc_forecast['gdp_growth'].point[-1]:.3f}%")
print(f"Recession GDP Q+8: {cond_recession['gdp_growth'].point[-1]:.3f}%")
print(f"Max GDP impact: {impact_df['gdp_growth'].min():.3f} pp")
print(f"Unemployment imposed: {last_unemp:.2f}% -> {recession_path[-1]:.2f}%")

## Exercise 2: Condition on Brazilian SELIC rate path

Load the `macro_brazil.csv` dataset, estimate a VAR model, and create conditional
forecasts by imposing a SELIC rate path (e.g., gradual cuts from the current level).
Compare the impact on Brazilian GDP and inflation.

In [ ]:
# Exercise 2 - SOLUTION: Brazilian SELIC rate path conditioning

# Load Brazilian macro data
df_br = load_macro_brazil()
print(f"Brazil data shape: {df_br.shape}")
print(f"Columns: {list(df_br.columns)}")
print(f"Period: {df_br.index[0]} to {df_br.index[-1]}")
df_br.tail()

# Variables: gdp_growth, inflation, interest_rate, unemployment, exchange_rate

In [ ]:
# Estimate VAR(2) for Brazil
br_var_names = ["gdp_growth", "inflation", "interest_rate", "unemployment", "exchange_rate"]
endog_br = df_br[br_var_names].values

model_br = SimpleVAR(endog_br, p_order=2, var_names=br_var_names)
print(f"Brazil VAR({model_br.p_order}) with {model_br.k_vars} variables")
print(f"Observations: {endog_br.shape[0]}")

# Unconditional forecast as baseline
cf_br = ConditionalForecast(model_br, method="analytic")
steps_br = 12  # monthly data -> 12 months ahead

unc_br = cf_br.forecast(steps=steps_br, conditions=None, n_draws=1000, seed=42)

print("\nUnconditional Forecast (Brazil):")
unc_br_df = pd.DataFrame(
    {name: unc_br[name].point for name in br_var_names},
    index=[f"M+{h+1}" for h in range(steps_br)]
)
print(unc_br_df.round(3).to_string())

In [ ]:
# Impose SELIC rising path: +200bps total over 12 months
# This mimics a monetary tightening cycle
last_selic = df_br["interest_rate"].iloc[-1]
print(f"Last observed SELIC (interest_rate): {last_selic:.2f}%")

# Gradual increase: ~16.7bp per month for 12 months = +200bps total
selic_hiking_path = [last_selic + (2.0 / steps_br) * (h + 1) for h in range(steps_br)]
print(f"SELIC hiking path: {[round(x, 2) for x in selic_hiking_path]}")
print(f"Total SELIC increase: {selic_hiking_path[-1] - last_selic:.2f} pp")

# Conditional forecast: fix interest_rate to hiking path
cond_br = cf_br.forecast(
    steps=steps_br,
    conditions={"interest_rate": selic_hiking_path},
    n_draws=1000,
    seed=42,
)

# Compare unconditional vs conditional for Brazil
horizons_br = np.arange(1, steps_br + 1)

fig, axes = plt.subplots(2, 3, figsize=(18, 9))

for idx, var in enumerate(br_var_names):
    ax = axes[idx // 3, idx % 3]
    ax.plot(horizons_br, unc_br[var].point, "b-o", label="Baseline", markersize=3, linewidth=2)
    ax.plot(horizons_br, cond_br[var].point, "r-s", label="SELIC +200bps", markersize=3, linewidth=2)

    if cond_br[var].lower_80 is not None:
        ax.fill_between(horizons_br, cond_br[var].lower_80, cond_br[var].upper_80,
                        alpha=0.15, color="red")

    ax.set_title(var, fontsize=12)
    ax.set_xlabel("Horizon (months)")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

# Remove empty subplot if odd number of variables
if len(br_var_names) < 6:
    axes[1, 2].set_visible(False)

fig.suptitle("Brazil: Conditional Forecast with SELIC Hiking +200bps", fontsize=14)
fig.tight_layout()
plt.show()

# Impact summary
print("\nImpact of SELIC +200bps on Brazilian Economy:")
cond_br_df = pd.DataFrame(
    {name: cond_br[name].point for name in br_var_names},
    index=[f"M+{h+1}" for h in range(steps_br)]
)
impact_br = cond_br_df - unc_br_df
print(impact_br.round(3).to_string())

# Reference values
print("\n--- Reference Values ---")
print(f"SELIC path: {last_selic:.2f}% -> {selic_hiking_path[-1]:.2f}% (+{selic_hiking_path[-1]-last_selic:.2f} pp)")
print(f"Baseline GDP M+12: {unc_br['gdp_growth'].point[-1]:.3f}%")
print(f"Cond. GDP M+12:    {cond_br['gdp_growth'].point[-1]:.3f}%")
print(f"GDP impact M+12:   {impact_br['gdp_growth'].iloc[-1]:.3f} pp")
print(f"Baseline Inflation M+12: {unc_br['inflation'].point[-1]:.3f}%")
print(f"Cond. Inflation M+12:    {cond_br['inflation'].point[-1]:.3f}%")
print(f"Inflation impact M+12:   {impact_br['inflation'].iloc[-1]:.3f} pp")